In [ ]:
import os
import json
from openai import OpenAI


COT_PROMPT = """
作为助理，您的职责是深入探究问题，并经过系统而详尽的思考，最终提供精准的解决方案。
这需要您进行全面的分析、总结、探索、重新评估、反思、回溯和迭代，以形成深思熟虑的思维过程。
请将您的回答分为两个主要部分：思考过程和解决方案。

在“思考过程”部分，请使用以下格式详细描述您的推理过程：<begin_of_thought>{思考过程，步骤之间用“\n\n”分隔} <end_of_thought> 每个步骤都应包含详细的思考过程，例如分析问题、总结相关发现、集思广益提出新想法、验证当前步骤的准确性、修正错误以及回顾之前的步骤。
在“解决方案”部分，基于“思考过程”部分中的各种尝试、探索和反思，系统地呈现您认为正确的最终解决方案。解决方案应保持逻辑清晰、准确简洁的表达风格，并详细说明得出结论所需的必要步骤，格式如下：<begin_of_solution> {最终格式化、精确且清晰的解决方案} <end_of_solution> 

现在，请尝试根据以上指导原则解决以下问题："
"""

llm_client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1"
)

def request_chat(query):

    try:
        prompt = COT_PROMPT + query
    
        completion = llm_client.chat.completions.create(
            model="Qwen32B",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=4096,
            temperature=0.6,
            top_p=0.95
        )
        result = completion.choices[0].message.content
        return query, result
        
    except:
        return query, ""

In [ ]:
# 测试 1
query = "大瓶芒果汁容量为30盎司，每瓶售价2700比塞塔。小瓶容量为6盎司，每瓶售价600比塞塔。购买一大瓶而不是同样体积的小瓶可以节省多少比塞塔？"
_, res = reque

In [ ]:
# 测试 2
query = "证明：如果A和B是两个n阶方阵，且AB=BA，那么tr(AB)=tr(BA)。"
_, res = request_chat(query)
print(res)

In [ ]:
# 测试 3
query = "写一篇小红书风格的帖子，300字以内，标题是🍑科颜氏男士护肤系列使用心得"
_, res = request_chat(query)
print(res)

In [ ]:
import re

pattern = r'<begin_of_thought>(.*?)<end_of_thought>'
reasoning = re.findall(pattern, res, re.DOTALL)
print("==== 思考的内容 ====")
print(reasoning[0])

pattern = r'<begin_of_solution>(.*?)<end_of_solution>'
answer = re.findall(pattern, res, re.DOTALL)
print("==== 答案 ====")
print(answer[0])

In [ ]:
import json
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed 

max_workers = 64

fd = open("./data/liucong/Chinese-DeepSeek-R1-Distill-data-110k/distill_r1_110k.jsonl")
inputs = []
for line in fd:
    inputs.append(json.loads(line)["input"])

# 测试128条
inputs = inputs[:128]

# 使用多进程生成 COT Reasoning PATH
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(request_chat, query) for query in inputs]
    for future in tqdm(as_completed(futures), total=len(inputs), desc="COT Generations"):
        query, cot_res = future.result()
        if cot_res:
            # write to file...
            pass